In [2]:
# %% [markdown]
# # Collapse Audit
#
# Standalone. Runs after every model's training notebooks have completed.
# Reads each model's saved VALIDATION predictions (part="val") and applies
# one identical collapse check across all of them — Ridge, Polymodel,
# Dense MLP, Sparse MLP, Dense KAN, Sparse KAN — rather than having some
# notebooks check inline and others not.
#
# Deliberately post-hoc, not inline, and deliberately detect-only, no retry:
# the collapse-handling decision is exclude-and-flag. A retry would require
# re-training with a different seed, which was explicitly ruled out — so
# there is no benefit to real-time detection during training. A single
# post-hoc pass over already-saved predictions is sufficient.
#
# Writes one flat CSV: collapse_audit.csv, with a `collapsed` boolean per
# (model, dataset, split, target_type, seed) row. This file must be
# merged against each model's own cross-seed summary table and filtered
# BEFORE computing any mean/std — otherwise a flagged run still silently
# pollutes the aggregate it was meant to be excluded from. See the
# `apply_collapse_filter` helper at the bottom of this notebook for the
# merge/query pattern to paste into each model notebook's summary cell.

# %%
from pathlib import Path
import pandas as pd
import numpy as np
import sys

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

sys.path.insert(0, "/content/drive/MyDrive/Thesis/Code")
from evaluation import check_collapse, load_predictions

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — confirm these match the actual saved folder layout
# ═══════════════════════════════════════════════════════════════════════════════

RESULTS_ROOT = Path("/content/drive/MyDrive/Thesis/Data/Results/Dense_vs_Sparse_KAN_v2")

# seeds=[None] means "no seed subfolder, read directly from the model's
# RESULTS_DIR" -- this is Ridge/Polymodel's layout (single deterministic
# fit, no seed loop). seeds=[42,123,456] means "read from seed_{n}/
# subfolders" -- this is every neural model's layout (3-seed reproducibility).
MODELS = {
    "ridge":       {"seeds": [None],            "datasets": ["agg_means", "agg_full_moments"]},
    "polymodel":   {"seeds": [None],            "datasets": ["agg_means", "agg_full_moments"]},
    "dense_mlp":   {"seeds": [42, 123, 456],    "datasets": ["agg_means", "agg_full_moments"]},
    "sparse_mlp":  {"seeds": [42, 123, 456],    "datasets": ["agg_means", "agg_full_moments"]},
    "sparse_mlp_v2":  {"seeds": [42, 123, 456],    "datasets": ["agg_means", "agg_full_moments"]},
    "dense_kan":   {"seeds": [42, 123, 456],    "datasets": ["agg_means", "agg_full_moments"]},
    "sparse_kan":  {"seeds": [42, 123, 456],    "datasets": ["agg_means", "agg_full_moments"]},
    "sparse_kan_v2":  {"seeds": [42, 123, 456],    "datasets": ["agg_means", "agg_full_moments"]},
    "sparse_kan_v3":  {"seeds": [42, 123, 456],    "datasets": ["agg_means", "agg_full_moments"]},
}
TARGET_TYPES = ["binary", "continuous"]
ALL_SPLITS   = ["Split_A", "Split_B", "Split_C", "Split_D"]

# %% [markdown]
# ## Run the Audit

# %%
rows = []
n_missing = 0

for model_key, cfg in MODELS.items():
    model_dir = RESULTS_ROOT / model_key
    if not model_dir.exists():
        print(f"  {model_key}: directory not found, skipping entirely")
        continue

    for seed in cfg["seeds"]:
        results_dir = model_dir / f"seed_{seed}" if seed is not None else model_dir

        for dataset in cfg["datasets"]:
            model_name = f"{model_key}_{dataset}"

            for target_type in TARGET_TYPES:
                for split in ALL_SPLITS:
                    try:
                        loaded = load_predictions(
                            model_name=model_name,
                            split_name=split,
                            target_type=target_type,
                            part="val",
                            results_dir=results_dir,
                        )
                        preds = loaded["predictions"]
                        signal = (preds["y_prob"].values if target_type == "binary"
                                 else preds["y_pred"].values)

                        result = check_collapse(signal)
                        rows.append({
                            "model": model_key, "dataset": dataset,
                            "target_type": target_type, "split": split,
                            "seed": seed, **result,
                        })
                    except FileNotFoundError:
                        n_missing += 1
                        continue   # config not yet run / not saved -- not an error

audit_df = pd.DataFrame(rows)

print(f"\n  Checked {len(audit_df)} runs across {audit_df['model'].nunique() if len(audit_df) else 0} models")
print(f"  Skipped (not found): {n_missing}")

if len(audit_df):
    n_collapsed = audit_df["collapsed"].sum()
    print(f"  Collapsed: {n_collapsed} ({n_collapsed / len(audit_df):.1%})")

    if n_collapsed:
        print("\n  COLLAPSED RUNS:")
        print(audit_df[audit_df.collapsed][
            ["model", "dataset", "target_type", "split", "seed", "pred_std"]
        ].to_string(index=False))
    else:
        print("\n  No collapsed runs detected.")

# %% [markdown]
# ## Save

# %%
audit_path = RESULTS_ROOT / "collapse_audit.csv"
audit_df.to_csv(audit_path, index=False)
print(f"\n  saved -> {audit_path}")

# %% [markdown]
# ## Per-Model / Per-Target Collapse Rate Summary
#
# Quick breakdown of WHERE collapses concentrate, if any -- e.g. confirming
# the earlier finding that binary is more prone to this than continuous.

# %%
if len(audit_df):
    summary = (audit_df.groupby(["model", "target_type"])["collapsed"]
              .agg(["sum", "count"]).reset_index())
    summary["rate"] = summary["sum"] / summary["count"]
    summary = summary.rename(columns={"sum": "n_collapsed", "count": "n_total"})
    print(summary.to_string(index=False))

# %% [markdown]
# ## Helper: apply this filter in each model's own cross-seed summary cell
#
# Paste this pattern into Dense MLP / Sparse MLP / Dense KAN / Sparse KAN's
# existing "Cross-Seed Summary" cell, BEFORE the groupby(...).agg(["mean","std"])
# calls, so a flagged run is excluded from every reported mean/std rather
# than silently averaged in.

# %%
def apply_collapse_filter(results_df: pd.DataFrame, audit_df: pd.DataFrame,
                          model_key: str) -> pd.DataFrame:
    """
    Left-merge a model's own results_df (one row per seed/dataset/split/target,
    as already built by that notebook's main experiment loop) against the
    collapse audit, and drop any row flagged collapsed=True.

    results_df must have columns: seed, dataset (or feature_set), split, target
    (rename feature_set -> dataset first if the notebook still uses the old name).
    """
    sub_audit = audit_df[audit_df.model == model_key][
        ["dataset", "target_type", "split", "seed", "collapsed"]
    ].rename(columns={"target_type": "target"})

    merged = results_df.merge(
        sub_audit, on=["dataset", "target", "split", "seed"], how="left"
    )
    # A row with no audit match (e.g. audit run before this model finished)
    # is treated as NOT collapsed rather than silently dropped -- fillna(False)
    merged["collapsed"] = merged["collapsed"].fillna(False)

    n_dropped = merged["collapsed"].sum()
    if n_dropped:
        print(f"  Excluding {n_dropped} collapsed run(s) from aggregation")

    return merged[~merged["collapsed"]].drop(columns="collapsed")


# Usage inside e.g. dense_kan's Cross-Seed Summary cell:
#
#   audit_df = pd.read_csv(RESULTS_ROOT / "collapse_audit.csv")
#   results_df = apply_collapse_filter(results_df, audit_df, model_key="dense_kan")
#   # ... proceed with the existing groupby(...).agg(["mean", "std"]) calls

Mounted at /content/drive


ModuleNotFoundError: No module named 'evaluation'